<a id="top"></a>

# Lab L0405: wire a tool to a model call

```yaml
title: "Lab L0405: wire a tool to a model call"
keywords: tool calling, tool_use, tool_result, dispatch, round-trip, raw anthropic sdk, lab
estimated duration: 35
```

> **Lesson:** L04. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L04/objectives.md). Solutions: `L0405_lab_solutions.ipynb`. Makes **live** Claude calls — set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) before running.
>
> **Why the raw Anthropic SDK, not `potato_llm`?** The course seam is text-only and cannot carry `tool_use`/`tool_result` blocks — L04 is about exactly those, so this lab calls the raw SDK (the key still loads through the config seam). See the L0403–L0404 demos.
>
> **After this lab you can:** pass a tool definition to a model, dispatch the returned `tool_use` block to a real function, and hand the result back to get a final answer.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Send the first turn and find the tool call](#2-problem-1--send-the-first-turn-and-find-the-tool-call)
- [3. Problem 2 — Dispatch: run the real function](#3-problem-2--dispatch-run-the-real-function)
- [4. Problem 3 — Continue: hand the result back](#4-problem-3--continue-hand-the-result-back)
- [5. Problem 4 — Why re-send the tool definition? (written)](#5-problem-4--why-re-send-the-tool-definition-written)
- [6. Self-check](#6-self-check)

## 1. Setup

The raw client, the single `calculator` tool and its definition (given), and a prompt.

In [ ]:
import anthropic

from fluffy_potato_curriculum.common.config import require_anthropic_key

MODEL = "claude-sonnet-4-6"
client = anthropic.Anthropic(api_key=require_anthropic_key())


def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, or raise ValueError on non-arithmetic input."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    return str(eval(expression))


CALCULATOR_TOOL: dict[str, object] = {
    "name": "calculator",
    "description": (
        "Evaluate a basic arithmetic expression (the four operators and parentheses) and "
        "return the exact numeric result. Use this whenever the user asks for a calculation."
    ),
    "input_schema": {
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "e.g. '18374 * 92431'."}},
        "required": ["expression"],
    },
}


PROMPT = "What is 6,150 multiplied by 7,012?"
print("setup ready (live model:", MODEL, ")")

[↑ Back to top](#top)

## 2. Problem 1 — Send the first turn and find the tool call

Send `PROMPT` with `tools=[CALCULATOR_TOOL]`. From the response, pull out the `tool_use` block (the block whose `.type == "tool_use"`). Print its `name`, `input`, and `id`.

In [ ]:
messages: list[dict[str, object]] = [{"role": "user", "content": PROMPT}]
# TODO: call client.messages.create(model=MODEL, max_tokens=400, tools=[CALCULATOR_TOOL],
#       messages=messages); then find the block with .type == 'tool_use'.
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 3. Problem 2 — Dispatch: run the real function

Check the tool name is `calculator`, pull the `expression` argument from `tool_use.input`, call `calculator(...)`, and capture the result string. This is the application doing the work the model only *proposed*.

In [ ]:
# TODO: assert the tool name, read tool_use.input['expression'], call calculator(...).
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 4. Problem 3 — Continue: hand the result back

Append the assistant's tool-use turn, then a **user-role** message carrying a `tool_result` block that references `tool_use.id` and contains your `result`. Send the full conversation again (tools included) and print the model's final text answer.

In [ ]:
# TODO: append assistant turn (first.content); append a user message whose content is a
#       [{'type':'tool_result','tool_use_id': tool_use.id, 'content': result}] block;
#       call messages.create again; join the text blocks of the reply.
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 5. Problem 4 — Why re-send the tool definition? (written)

In Problem 3 you passed `tools=[CALCULATOR_TOOL]` again on the *second* call. In a sentence or two: why is the definition required on every request, not just the first?

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 6. Self-check

Compare against `L0405_lab_solutions.ipynb`. Done when the model proposes a `calculator` call, your dispatch returns `43,123,800`, and the final answer uses that number. You can state why the tool definition rides along on every call (the model is stateless).

[↑ Back to top](#top)